# Pommerman FFA — Example Notebook

Demonstrates:
1. Installation check.
2. Random rollout with 4 RandomAgents to validate the raw env.
3. Learner-vs-3-SimpleAgent wrapper rollout.
4. IQL training (1 learner vs 3 built-in bots).
5. Reward curve plot.

## 0. Install Pommerman (run once)

```bash
git clone https://github.com/MultiAgentLearning/playground ~/playground
cd ~/playground
sed -i 's/gym==[0-9\.]*/gym/' setup.py   # remove old gym version pin
pip install -e .
```

Then restart the kernel and continue.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import pommerman
    print('Pommerman version:', pommerman.__version__)
except ImportError:
    print('Pommerman not installed — see the Install section above.')
    raise

## 1. Random rollout with 4 RandomAgents

In [ ]:
from pommerman import agents

agent_list = [agents.RandomAgent() for _ in range(4)]
raw_env = pommerman.make('PommeFFACompetition-v0', agent_list)

for ep in range(2):
    obs = raw_env.reset()
    done = False
    steps = 0
    total_rewards = [0.0] * 4
    while not done:
        actions = raw_env.act(obs)
        obs, rewards, done, info = raw_env.step(actions)
        for i, r in enumerate(rewards):
            total_rewards[i] += r
        steps += 1
    print(f'Episode {ep}: {steps} steps | rewards={total_rewards}')

raw_env.close()

## 2. Learner-vs-3-SimpleAgent wrapper

In [ ]:
from discrete_action_space.pommerman_ffa import make_pz_env
from discrete_action_space.marl_utils import random_rollout

env = make_pz_env(learner_slot=0)
print('Agents      :', env.possible_agents)

# One manual step to confirm obs space
obs, _ = env.reset()
print('Obs shape   :', obs['learner_0'].shape)
print('Action space:', env.action_space('learner_0'))

print('=== Random rollout (learner-vs-SimpleAgents) ===')
random_rollout(env, n_episodes=2)
env.close()

## 3. IQL Training

In [ ]:
from discrete_action_space.marl_utils import run_iql

def make_env():
    from discrete_action_space.pommerman_ffa import make_pz_env
    return make_pz_env(learner_slot=0)

# Train for 20k frames (increase for real training)
results = run_iql(
    make_env_fn=make_env,
    n_frames=20_000,
    frames_per_batch=500,
    train_batch_size=64,
    memory_size=20_000,
    save_folder='checkpoints/pommerman_iql',
)
print('Training complete.')

## 4. Reward curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ep_rewards = results['episode_rewards']
fig, ax = plt.subplots(figsize=(10, 4))
for agent, rews in ep_rewards.items():
    window = 20
    rolling = np.convolve(rews, np.ones(window) / window, mode='valid')
    ax.plot(rolling, label=agent)
ax.set_xlabel('Episode')
ax.set_ylabel(f'Reward (rolling avg {window})')
ax.set_title('Pommerman FFA — IQL learner vs 3 SimpleAgents')
ax.legend()
plt.tight_layout()
plt.savefig('checkpoints/pommerman_iql/training_curve.png', dpi=120)
plt.show()